# RIME-X Complete Pipeline
### Steps 1–4: Install → GMT → Warming Levels → Quantile Maps → Emulator Output

**Pipeline overview:**
```
CMIP6 Regional CSVs (already done)
        ↓
Step 1: Install rimeX
        ↓
Step 2: Setup config.toml (tell rimeX where your data lives)
        ↓
Step 3: rime-pre-gmt  → global mean temperature timeseries
        ↓
Step 4: rime-pre-wl   → warming level table
        ↓
Step 5: rime-pre-quantilemap → quantile maps (core of RIME-X)
        ↓
Step 6: Run emulator with MAGICC → probabilistic regional projections
```


## Step 1 — Install rimeX

In [ ]:
# Install rimeX from GitHub (development install — recommended)
# Run this cell ONCE, then restart kernel if needed

import subprocess, sys

result = subprocess.run(
    ["pip", "install", "-e", ".", "--quiet"],
    capture_output=True, text=True,
    cwd="./rimeX"   # path to cloned repo — update if different
)

# If you haven't cloned it yet, uncomment and run these first:
# !git clone https://github.com/iiasa/rimeX.git
# then re-run this cell

print(result.stdout)
print(result.stderr)

# Verify installation
try:
    import rimeX
    print("rimeX installed successfully!")
except ImportError:
    print("rimeX not found — check path to cloned repo above")


In [ ]:
# Verify all CLI tools are available
import subprocess

tools = [
    "rime-pre-gmt",
    "rime-pre-wl",
    "rime-pre-quantilemap",
    "rime-config",
]

print("Checking CLI tools:")
for tool in tools:
    result = subprocess.run(["which", tool], capture_output=True, text=True)
    status = "✅ found" if result.returncode == 0 else "❌ NOT FOUND"
    print(f"  {tool:30s} {status}")


## Step 2 — Setup config.toml

In [ ]:
# ── PATHS — update these to match your folder structure ──────────────

import os

# Root of your project (where GCM_models/ lives)
PROJECT_ROOT = os.path.abspath(".")

# Where your regional average CSVs were written (from the previous pipeline)
REGIONAL_DATA_DIR = f"{PROJECT_ROOT}/GCM_models/CNRM-ESM2-1/regional_averages_output"

# Where global mean CSVs are
GMT_DIR = f"{REGIONAL_DATA_DIR}/cmip-6_global_mean/tas"

# Where RIME-X will write its outputs
RIMEX_OUTPUT_DIR = f"{PROJECT_ROOT}/rimex_output"
WARMING_LEVELS_DIR = f"{RIMEX_OUTPUT_DIR}/warming_levels"
QUANTILE_MAPS_DIR  = f"{RIMEX_OUTPUT_DIR}/quantile_maps"

# Your MAGICC CSV (the cleaned 600-ensemble file)
MAGICC_CSV = f"{PROJECT_ROOT}/ssp245_MAGICC_separated.csv"

# Create output directories
for d in [RIMEX_OUTPUT_DIR, WARMING_LEVELS_DIR, QUANTILE_MAPS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Paths configured:")
print(f"  Regional data   : {REGIONAL_DATA_DIR}")
print(f"  GMT CSVs        : {GMT_DIR}")
print(f"  MAGICC CSV      : {MAGICC_CSV}")
print(f"  Output root     : {RIMEX_OUTPUT_DIR}")
print(f"  Warming levels  : {WARMING_LEVELS_DIR}")
print(f"  Quantile maps   : {QUANTILE_MAPS_DIR}")
print()
print("Path checks:")
print(f"  Regional data exists : {os.path.exists(REGIONAL_DATA_DIR)}")
print(f"  GMT dir exists       : {os.path.exists(GMT_DIR)}")
print(f"  MAGICC CSV exists    : {os.path.exists(MAGICC_CSV)}")


In [ ]:
# Write config.toml — tells rimeX where all data lives
# rimeX reads this automatically from the working directory

config_content = f"""
# RIME-X configuration file
# Place this in your project root OR pass --config flag to CLI tools

[indicators]
folder = "{REGIONAL_DATA_DIR}/isimip_regional_data"

[isimip]
climate_impact_explorer = "{RIMEX_OUTPUT_DIR}"
download_folder = "{PROJECT_ROOT}/GCM_models"

[preprocessing.regional]
weights = ["latWeight"]
masks_folder = "{PROJECT_ROOT}/masks"

[preprocessing]
warming_level_step = 0.1
warming_level_min  = 1.0
running_mean_window = 21
quantilemap_quantile_bins = 101

[output]
folder = "{RIMEX_OUTPUT_DIR}"
"""

config_path = f"{PROJECT_ROOT}/config.toml"
with open(config_path, "w") as f:
    f.write(config_content)

print(f"config.toml written to: {config_path}")
print()
print("Contents:")
print(config_content)


## Step 3 — Global Mean Temperature (`rime-pre-gmt`)

In [ ]:
# List the GMT CSV files you already have
import os, pandas as pd

gmt_files = [f for f in os.listdir(GMT_DIR) if f.endswith(".csv")]
print(f"GMT CSV files found: {len(gmt_files)}")
for f in gmt_files:
    df = pd.read_csv(f"{GMT_DIR}/{f}")
    print(f"  {f}")
    print(f"    rows={len(df)}, years={df['time'].iloc[0]} to {df['time'].iloc[-1]}")
    print(f"    tas range: {df['tas'].min():.2f} to {df['tas'].max():.2f} K")


In [ ]:
# Run rime-pre-gmt to compute 21-year running mean GMT
# (rimeX uses this running mean to assign warming levels)

import subprocess

gmt_file = f"{GMT_DIR}/{gmt_files[0]}"   # use first available GMT file

result = subprocess.run(
    [
        "rime-pre-gmt",
        "--input",  gmt_file,
        "--output", WARMING_LEVELS_DIR,
        "--running-mean-window", "21",
    ],
    capture_output=True, text=True
)

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)

if result.returncode == 0:
    print("\n✅ rime-pre-gmt completed successfully!")
else:
    print("\n❌ Error — check STDERR above")
    print("Tip: run  rime-pre-gmt --help  to see all options")


## Step 4 — Warming Level Table (`rime-pre-wl`)

In [ ]:
# rime-pre-wl scans all CMIP6 model runs and assigns each year
# to a warming level bin (e.g. 1.0C, 1.1C, 1.2C ... 4.0C)
# This is what connects GMT trajectory → regional response

import subprocess

result = subprocess.run(
    [
        "rime-pre-wl",
        "--gmt-folder",  WARMING_LEVELS_DIR,
        "--output",      WARMING_LEVELS_DIR,
        "--step",        "0.1",    # 0.1C bins
        "--min",         "1.0",    # start from 1.0C
    ],
    capture_output=True, text=True
)

print("STDOUT:", result.stdout)
print("STDERR:", result.stderr)
print("Return code:", result.returncode)

if result.returncode == 0:
    print("\n✅ rime-pre-wl completed successfully!")
    # Show what was produced
    wl_files = os.listdir(WARMING_LEVELS_DIR)
    print(f"Files in warming_levels/: {wl_files}")
else:
    print("\n❌ Error — run  rime-pre-wl --help  to see all options")


In [ ]:
# Preview the warming level table
import pandas as pd, os

wl_files = [f for f in os.listdir(WARMING_LEVELS_DIR) if f.endswith(".csv")]
if wl_files:
    df = pd.read_csv(f"{WARMING_LEVELS_DIR}/{wl_files[0]}")
    print(f"Warming level file: {wl_files[0]}")
    print(f"Shape: {df.shape}")
    print(df.head(10).to_string(index=False))
else:
    print("No CSV files yet in warming_levels/ — check Step 4 output")


## Step 5 — Build Quantile Maps (`rime-pre-quantilemap`)

In [ ]:
# Quantile maps are the CORE of RIME-X
# For each warming level (1.0C, 1.1C ... 4.0C) and each region,
# they store the full distribution (101 quantiles) of the indicator
# This is what makes RIME-X a proper probabilistic emulator

# We build regional quantile maps (one NetCDF per region)
import subprocess

# Get list of available regions from your isimip_regional_data folder
regional_dir = f"{REGIONAL_DATA_DIR}/isimip_regional_data"
available_regions = os.listdir(regional_dir) if os.path.exists(regional_dir) else []
print(f"Regions available: {len(available_regions)}")
print(f"Sample: {available_regions[:10]}")


In [ ]:
# Build quantile maps for all available regions
# --regional flag: one NetCDF per region (includes all sub-regions)
# --season annual: compute annual averages
# --quantile-bins 101: 101 quantiles (0%, 1%, 2% ... 100%)
# --weights latWeight: area-weighted averages

import subprocess

result = subprocess.run(
    [
        "rime-pre-quantilemap",
        "--regional",
        "--season",         "annual",
        "--quantile-bins",  "101",
        "--weights",        "latWeight",
        "--warming-levels", WARMING_LEVELS_DIR,
        "--input",          f"{REGIONAL_DATA_DIR}/isimip_regional_data",
        "--output",         QUANTILE_MAPS_DIR,
    ],
    capture_output=True, text=True
)

print("STDOUT:", result.stdout[-3000:])   # last 3000 chars to avoid overflow
print("STDERR:", result.stderr[-1000:])
print("Return code:", result.returncode)

if result.returncode == 0:
    print("\n✅ Quantile maps built successfully!")
    qm_files = os.listdir(QUANTILE_MAPS_DIR)
    print(f"Quantile map files: {len(qm_files)}")
    print(f"Sample: {qm_files[:5]}")
else:
    print("\n❌ Error — run  rime-pre-quantilemap --help  for all options")


In [ ]:
# Preview a quantile map NetCDF
import xarray as xr, os

qm_files = [f for f in os.listdir(QUANTILE_MAPS_DIR) if f.endswith(".nc")]

if qm_files:
    sample_qm = f"{QUANTILE_MAPS_DIR}/{qm_files[0]}"
    ds = xr.open_dataset(sample_qm)
    print(f"Quantile map: {qm_files[0]}")
    print(f"\nDataset structure:")
    print(ds)
    print(f"\nDimensions  : {dict(ds.dims)}")
    print(f"Variables   : {list(ds.data_vars)}")
    print(f"Coordinates : {list(ds.coords)}")
    ds.close()
else:
    print("No .nc files found in quantile_maps/ yet")


## Step 6 — Run Emulator with MAGICC Ensembles

In [ ]:
# Load your cleaned MAGICC 600-ensemble GMT file
import pandas as pd

gmt_magicc = pd.read_csv(MAGICC_CSV, index_col="year")
print(f"MAGICC GMT shape : {gmt_magicc.shape}")
print(f"Years            : {gmt_magicc.index.min()} to {gmt_magicc.index.max()}")
print(f"Ensembles        : {len(gmt_magicc.columns)} (columns: {list(gmt_magicc.columns[:5])}...)")
print()
print("Preview (first 3 years, first 5 ensembles):")
print(gmt_magicc.iloc[:3, :5].to_string())


In [ ]:
# Run RIME-X emulator using quantile maps + MAGICC GMT
# This produces probabilistic regional climate projections
#
# For each region, for each year:
#   - Each of 600 MAGICC ensembles gives a GMT value
#   - That GMT is looked up in the quantile map
#   - A regional indicator value is drawn from the corresponding distribution
#   - Result: 600 regional trajectories = full uncertainty range

from rimeX.preproc.quantilemaps import make_quantilemap_prediction, get_filepath
import xarray as xr
import pandas as pd
import os

# ── Configure which indicator / region / season to emulate ───────────
INDICATOR   = "tas"        # variable name
REGION      = "PAK"        # country/region code
SUBREGION   = "PAK"        # sub-region (same as REGION for country-level)
SEASON      = "annual"     # annual / summer / winter / spring / autumn
WEIGHT      = "latWeight"  # latWeight / pop2020 / gdp2020
QUANTILES   = [0.05, 0.17, 0.5, 0.83, 0.95]   # 5th, 17th, median, 83rd, 95th
N_SAMPLES   = 5000         # Monte Carlo samples for uncertainty

print(f"Running emulator for:")
print(f"  Indicator  : {INDICATOR}")
print(f"  Region     : {REGION} / {SUBREGION}")
print(f"  Season     : {SEASON}")
print(f"  Quantiles  : {QUANTILES}")


In [ ]:
# Load quantile map for the selected region
import xarray as xr

qm_file = f"{QUANTILE_MAPS_DIR}/{INDICATOR}_{REGION}_{SEASON}_{WEIGHT}_eq.nc"

if not os.path.exists(qm_file):
    # Try to find the file with a different naming pattern
    available = [f for f in os.listdir(QUANTILE_MAPS_DIR) if REGION in f and f.endswith(".nc")]
    print(f"Expected file not found: {qm_file}")
    print(f"Available files with '{REGION}': {available}")
else:
    with xr.open_dataset(qm_file) as ds:
        impact_data = ds[INDICATOR].sel(region=SUBREGION).load()

    print(f"Quantile map loaded: {qm_file}")
    print(f"Shape     : {impact_data.shape}")
    print(f"Dimensions: {dict(impact_data.dims)}")
    print(f"Warming levels: {impact_data.coords['warming_level'].values[:5]}...")
    print(f"Quantiles     : {impact_data.coords['quantile'].values[:5]}...")


In [ ]:
# Run the emulator — produces 600 regional trajectories
from rimeX.preproc.quantilemaps import make_quantilemap_prediction

results = make_quantilemap_prediction(
    impact_data,
    gmt_magicc,
    quantiles = QUANTILES,
    samples   = N_SAMPLES,
    clip      = True,
    seed      = 42,
)

print(f"Emulator output shape : {results.shape}")
print(f"Dimensions            : {results.dims}")
print()
print("Preview (first 5 years, all quantiles):")
print(results.to_pandas().head().to_string())

# Save output
output_csv = f"{RIMEX_OUTPUT_DIR}/emulator_output_{INDICATOR}_{REGION}_{SEASON}.csv"
results.T.to_pandas().to_csv(output_csv)
print(f"\n✅ Saved to: {output_csv}")


## Step 7 — Visualize Probabilistic Output

In [ ]:
# Plot the probabilistic regional projections
import matplotlib.pyplot as plt
import pandas as pd

output_csv = f"{RIMEX_OUTPUT_DIR}/emulator_output_{INDICATOR}_{REGION}_{SEASON}.csv"
df = pd.read_csv(output_csv, index_col=0)
df.index = df.index.astype(int)

fig, ax = plt.subplots(figsize=(12, 6))

# Shade uncertainty ranges
ax.fill_between(df.index, df["0.05"], df["0.95"],
                alpha=0.2, color="steelblue", label="5–95th percentile")
ax.fill_between(df.index, df["0.17"], df["0.83"],
                alpha=0.35, color="steelblue", label="17–83rd percentile")

# Median line
ax.plot(df.index, df["0.5"],
        color="steelblue", linewidth=2, label="Median (50th)")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel(f"{INDICATOR} anomaly (K)", fontsize=12)
ax.set_title(
    f"RIME-X Probabilistic Projection\n"
    f"{INDICATOR.upper()} | {REGION} | {SEASON} | SSP2-4.5 | 600 MAGICC ensembles",
    fontsize=13
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")

plt.tight_layout()
plot_path = f"{RIMEX_OUTPUT_DIR}/emulator_output_{INDICATOR}_{REGION}_{SEASON}.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved to: {plot_path}")
